In [1]:
import pickle
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("Loading data and features...")

Loading data and features...


In [2]:
DATA_DIR = Path("/kaggle/input/dataset")
FEATURE_DIR = Path("/kaggle/input/features")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Train: (75000, 4), Test: (75000, 3)
Using device: cuda
GPU: Tesla T4


In [3]:
# Winsorization
lower_cap = train_df['price'].quantile(0.01)
upper_cap = train_df['price'].quantile(0.99)
train_df['price'] = train_df['price'].clip(lower=lower_cap, upper=upper_cap)

print(f"Winsorized price range: ${lower_cap:.2f} to ${upper_cap:.2f}")
print(f"Target stats: mean={train_df['price'].mean():.2f}, std={train_df['price'].std():.2f}")

Winsorized price range: $1.32 to $145.25
Target stats: mean=22.87, std=25.63


In [4]:
# Extract features
def extract_basic_features(df):
    df = df.copy()
    
    df['pack_quantity'] = df['catalog_content'].str.extract(r'Pack of (\d+)', expand=False).fillna('1').astype(float)
    
    weight_match = df['catalog_content'].str.extract(r'(\d+\.?\d*)\s*(oz|ounce|pound|lb|kg|g\b)', expand=True)
    df['weight_oz'] = pd.to_numeric(weight_match[0], errors='coerce').fillna(0)
    
    volume_match = df['catalog_content'].str.extract(r'(\d+\.?\d*)\s*(fl oz|ml|liter|gallon)', expand=True)
    df['volume_floz'] = pd.to_numeric(volume_match[0], errors='coerce').fillna(0)
    
    dims = df['catalog_content'].str.extract(r'(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*x\s*(\d+\.?\d*)', expand=True)
    df['dim1'] = pd.to_numeric(dims[0], errors='coerce').fillna(0)
    df['dim2'] = pd.to_numeric(dims[1], errors='coerce').fillna(0)
    df['dim3'] = pd.to_numeric(dims[2], errors='coerce').fillna(0)
    df['volume_calc'] = df['dim1'] * df['dim2'] * df['dim3']
    
    brand_match = df['catalog_content'].str.extract(r'Brand:\s*([^\n\|]+)', expand=False)
    df['brand_clean'] = brand_match.fillna('unknown').str.lower().str.strip()
    
    feature_cols = ['pack_quantity', 'weight_oz', 'volume_floz', 'dim1', 'dim2', 'dim3', 'volume_calc']
    return df, feature_cols

train_df, basic_cols = extract_basic_features(train_df)
test_df, _ = extract_basic_features(test_df)

# Target encoding
def target_encode_brand(train_df, test_df):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    train_df['brand_target_enc'] = 0.0
    global_mean = train_df['price'].mean()
    
    for train_idx, val_idx in kf.split(train_df):
        train_fold = train_df.iloc[train_idx]
        brand_means = train_fold.groupby('brand_clean')['price'].mean()
        train_df.loc[val_idx, 'brand_target_enc'] = train_df.iloc[val_idx]['brand_clean'].map(brand_means).fillna(global_mean)
    
    brand_means_full = train_df.groupby('brand_clean')['price'].mean()
    test_df['brand_target_enc'] = test_df['brand_clean'].map(brand_means_full).fillna(global_mean)
    
    return train_df, test_df

train_df, test_df = target_encode_brand(train_df, test_df)

# Interactions
for df in [train_df, test_df]:
    df['pack_squared'] = df['pack_quantity'] ** 2
    df['pack_sqrt'] = np.sqrt(df['pack_quantity'])
    df['pack_log'] = np.log1p(df['pack_quantity'])
    df['weight_per_unit'] = df['weight_oz'] / df['pack_quantity'].clip(lower=1)
    df['volume_per_unit'] = df['volume_floz'] / df['pack_quantity'].clip(lower=1)

interaction_cols = ['pack_squared', 'pack_sqrt', 'pack_log', 'weight_per_unit', 'volume_per_unit']
feature_cols_handcrafted = basic_cols + ['brand_target_enc'] + interaction_cols

print(f"Handcrafted features: {len(feature_cols_handcrafted)}")

Handcrafted features: 13


In [5]:
# Load pre-computed features
text_features_train = np.load(FEATURE_DIR / "text_features_enhanced_train.npy")
text_features_test = np.load(FEATURE_DIR / "text_features_enhanced_test.npy")
image_features_train = np.load(FEATURE_DIR / "image_features_train_full.npy")
image_features_test = np.load(FEATURE_DIR / "image_features_test_full.npy")

print(f"Text features: {text_features_train.shape}")
print(f"Image features: {image_features_train.shape}")

# Combine ALL features
X_handcrafted = train_df[feature_cols_handcrafted].values
X_train_combined = np.hstack([X_handcrafted, text_features_train, image_features_train])

X_handcrafted_test = test_df[feature_cols_handcrafted].values
X_test_combined = np.hstack([X_handcrafted_test, text_features_test, image_features_test])

y_train = train_df['price'].values

print(f"\nFinal feature matrix: {X_train_combined.shape}")
print(f"Total features: {X_train_combined.shape[1]}")

Text features: (75000, 802)
Image features: (75000, 512)

Final feature matrix: (75000, 1327)
Total features: 1327


In [6]:
# Standardize ALL features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_combined)
X_test_scaled = scaler.transform(X_test_combined)

print(f"Scaled features - Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Scaled features - Train: (75000, 1327), Test: (75000, 1327)


In [7]:
class PricePredictor(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128, 64], dropout=0.3):
        super(PricePredictor, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

# Model parameters
input_dim = X_train_scaled.shape[1]
model = PricePredictor(
    input_dim=input_dim,
    hidden_dims=[512, 256, 128, 64],
    dropout=0.3
).to(device)

print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Model architecture:
PricePredictor(
  (network): Sequential(
    (0): Linear(in_features=1327, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=64, bias=True)
    (13): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.3, inplace=False)
    (16): Linear(in_features=64, out_features=1, bias=True)
  )
)

Total parameters: 854,401


In [8]:
def smape_loss(y_pred, y_true):
    """SMAPE loss for neural network training"""
    denominator = (torch.abs(y_true) + torch.abs(y_pred)) / 2.0
    diff = torch.abs(y_true - y_pred)
    return torch.mean(diff / (denominator + 1e-8))

def smape_metric(y_true, y_pred):
    """SMAPE metric for evaluation (numpy)"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    return np.mean(diff / denominator) * 100

print("SMAPE loss and metric defined")

SMAPE loss and metric defined


In [9]:
# Training hyperparameters
BATCH_SIZE = 512
EPOCHS = 100
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5

def train_fold(model, train_loader, val_loader, epochs, lr, device):
    """Train model for one fold"""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=False)
    
    best_val_smape = float('inf')
    patience_counter = 0
    patience_limit = 15
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X).squeeze()
            loss = smape_loss(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                outputs = model(batch_X).squeeze()
                val_preds.extend(outputs.cpu().numpy())
                val_true.extend(batch_y.numpy())
        
        val_preds = np.clip(val_preds, 0, None)
        val_smape = smape_metric(val_true, val_preds)
        
        scheduler.step(val_smape)
        
        if val_smape < best_val_smape:
            best_val_smape = val_smape
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{epochs}: Train Loss={train_loss/len(train_loader):.4f}, Val SMAPE={val_smape:.2f}, Best={best_val_smape:.2f}")
        
        if patience_counter >= patience_limit:
            print(f"    Early stopping at epoch {epoch+1}")
            break
    
    return best_val_smape

print(f"Training config: BS={BATCH_SIZE}, LR={LEARNING_RATE}, Epochs={EPOCHS}")

Training config: BS=512, LR=0.001, Epochs=100


In [10]:
# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_scores = []

print("\nStarting 5-fold cross-validation...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled), 1):
    print(f"{'='*60}")
    print(f"FOLD {fold}/5")
    print(f"{'='*60}")
    
    # Prepare data
    X_train_fold = torch.FloatTensor(X_train_scaled[train_idx])
    y_train_fold = torch.FloatTensor(y_train[train_idx])
    X_val_fold = torch.FloatTensor(X_train_scaled[val_idx])
    y_val_fold = torch.FloatTensor(y_train[val_idx])
    
    train_dataset = TensorDataset(X_train_fold, y_train_fold)
    val_dataset = TensorDataset(X_val_fold, y_val_fold)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Initialize fresh model
    model = PricePredictor(
        input_dim=input_dim,
        hidden_dims=[512, 256, 128, 64],
        dropout=0.3
    ).to(device)
    
    # Train
    best_smape = train_fold(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, device)
    fold_scores.append(best_smape)
    
    print(f"Fold {fold} Best SMAPE: {best_smape:.2f}\n")

print(f"\n{'='*60}")
print(f"NEURAL NETWORK CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.2f}")
print(f"\nMean CV SMAPE: {np.mean(fold_scores):.2f} ± {np.std(fold_scores):.2f}")
print(f"{'='*60}")


Starting 5-fold cross-validation...

FOLD 1/5
    Epoch 10/100: Train Loss=0.4668, Val SMAPE=51.56, Best=51.56
    Epoch 20/100: Train Loss=0.3765, Val SMAPE=49.52, Best=49.52
    Epoch 30/100: Train Loss=0.3278, Val SMAPE=48.99, Best=48.73
    Epoch 40/100: Train Loss=0.2947, Val SMAPE=49.27, Best=48.63
    Epoch 50/100: Train Loss=0.2499, Val SMAPE=48.15, Best=48.15
    Epoch 60/100: Train Loss=0.2274, Val SMAPE=48.07, Best=48.07
    Epoch 70/100: Train Loss=0.2155, Val SMAPE=48.07, Best=48.05
    Epoch 80/100: Train Loss=0.2086, Val SMAPE=48.10, Best=47.95
    Early stopping at epoch 87
  → Fold 1 Best SMAPE: 47.95

FOLD 2/5
    Epoch 10/100: Train Loss=0.4654, Val SMAPE=50.36, Best=50.36
    Epoch 20/100: Train Loss=0.3795, Val SMAPE=49.03, Best=48.56
    Epoch 30/100: Train Loss=0.3289, Val SMAPE=48.44, Best=48.08
    Epoch 40/100: Train Loss=0.2789, Val SMAPE=47.53, Best=47.53
    Epoch 50/100: Train Loss=0.2483, Val SMAPE=47.45, Best=47.35
    Epoch 60/100: Train Loss=0.2336, V

In [11]:
print("Training final model on full training data...")

# Prepare full training data
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize final model
final_model = PricePredictor(
    input_dim=input_dim,
    hidden_dims=[512, 256, 128, 64],
    dropout=0.2  # Lower dropout for final model
).to(device)

optimizer = optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Train for more epochs on full data
FINAL_EPOCHS = 80

for epoch in range(FINAL_EPOCHS):
    final_model.train()
    train_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = final_model(batch_X).squeeze()
        loss = smape_loss(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_loss = train_loss / len(train_loader)
    scheduler.step(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{FINAL_EPOCHS}: Loss={avg_loss:.4f}")

print("\nFinal model training complete!")

Training final model on full training data...
Epoch 10/80: Loss=0.4326
Epoch 20/80: Loss=0.3372
Epoch 30/80: Loss=0.2890
Epoch 40/80: Loss=0.2562
Epoch 50/80: Loss=0.2349
Epoch 60/80: Loss=0.2194
Epoch 70/80: Loss=0.2080
Epoch 80/80: Loss=0.1985

Final model training complete!


In [12]:
# Generate test predictions
final_model.eval()
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)

with torch.no_grad():
    test_predictions = final_model(X_test_tensor).squeeze().cpu().numpy()

# Clip predictions
test_predictions = np.clip(test_predictions, 0, None)

print(f"\nTest predictions shape: {test_predictions.shape}")
print(f"Prediction stats:")
print(f"  Mean: ${test_predictions.mean():.2f}")
print(f"  Median: ${np.median(test_predictions):.2f}")
print(f"  Min: ${test_predictions.min():.2f}")
print(f"  Max: ${test_predictions.max():.2f}")


Test predictions shape: (75000,)
Prediction stats:
  Mean: $19.46
  Median: $14.56
  Min: $1.48
  Max: $180.80


In [13]:
# Save predictions
output_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_predictions
})

output_df.to_csv('test_out_nn.csv', index=False)
print(f"\nPredictions saved to test_out_nn.csv")
print(f"Total predictions: {len(output_df):,}")


Predictions saved to test_out_nn.csv
Total predictions: 75,000
